<a href="https://colab.research.google.com/github/JuanZapa7a/AINavalEngineering/blob/main/NB03_NumPy_Array_Mechanics_Dtypes_Indexing_and_Views.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# **NB03 · Class 3 — NumPy Array Mechanics: dtypes, Indexing, and Views**

## Block 2: AI — Machine Learning (continued)

`NB02` used NumPy arrays for basic creation, indexing, and vectorized math — enough to get a first plot on screen, but not enough to use NumPy with real confidence. Every model this course trains, from `NB07`'s first regression through the deepest neural network, is built on arrays underneath. This class goes one level deeper on the array itself: what a `dtype` actually controls, the difference between a **view** and a **copy** (a real, common source of silent bugs), and fancy/boolean indexing patterns used constantly for real data.

This class works entirely with data already used elsewhere in this course — `sonar.all-data` (60 real acoustic features, from `NB08`) and `ship_fuel_efficiency.csv` (from `NB07`) — so every example is grounded in a real dataset, not toy numbers.

### Learning objectives

By the end of this class, students will be able to:
- Explain what a `dtype` is and why it matters for memory and correctness.
- Create arrays with the right shape and type for a given task (`zeros`, `ones`, `arange`, `linspace`, `meshgrid`, `identity`).
- Index and slice 1-D and multi-dimensional arrays confidently, including negative indices and steps.
- Explain, and demonstrate with real code, the difference between a NumPy **view** and a **copy** — and why it matters.
- Use boolean masks and fancy indexing to select real data by condition, not just by position.

### Agenda (2-hour class)

| # | Section | Minutes |
|---|---|---|
| 1 | Recap and why array mechanics matters | 5 |
| 2 | dtypes: what they are and why they matter | 15 |
| 3 | Array creation patterns | 15 |
| 4 | Indexing and slicing, 1-D and multi-D | 20 |
| 5 | Views vs. copies (a real, common bug) | 20 |
| 6 | Boolean masks and fancy indexing on real data | 25 |
| 7 | Summary, homework, next class | 20 |

As always: approximate guidance, not a script.


---

## 1. Why array mechanics matters


Every dataset in this course eventually becomes a NumPy array (directly, or inside a Pandas DataFrame, which wraps NumPy arrays column by column). A model that silently gets the wrong slice of data, or accidentally shares memory it shouldn't, produces wrong results **without an error** — `arguably worse than a crash, because nothing tells you it happened`. This class is about the mechanics that prevent that class of bug.

> **Further reading**: [NumPy array basics (Wikipedia)](https://en.wikipedia.org/wiki/NumPy) | [NumPy official documentation](https://numpy.org/doc/stable/)


---

## 2. dtypes: what they are and why they matter


Every NumPy array has a single `dtype` (data type) shared by every element — unlike a Python list, which can freely mix types. This is exactly what makes NumPy fast: knowing every element is, say, a 64-bit float lets NumPy operate on the whole array in tight, vectorized loops instead of checking each element's type one at a time.

The cell below loads the real Sonar dataset's 60 acoustic-frequency features and inspects its dtype directly.


In [ ]:
import numpy as np
import urllib.request

sonar_url = "https://raw.githubusercontent.com/JuanZapa7a/AINavalEngineering/main/Datasets/sonar.all-data"
raw_lines = urllib.request.urlopen(sonar_url).read().decode("utf-8").strip().split("\n")

sonar_features = np.array([[float(x) for x in line.split(",")[:-1]] for line in raw_lines])
sonar_labels = np.array([line.split(",")[-1] for line in raw_lines])

print(f"Shape: {sonar_features.shape}  (rows=readings, cols=frequency bands)")
print(f"dtype: {sonar_features.dtype}")
print(f"Memory used: {sonar_features.nbytes / 1024:.1f} KB")


The next cell demonstrates the real memory cost of dtype choice: the same 208x60 array of values, stored as 64-bit floats (NumPy's default) versus 32-bit floats. Halving precision genuinely halves memory — `a real, practical trade-off once a dataset is large enough for it to matter` (irrelevant here at ~100 KB, but very real for the multi-GB grids `NB18`/`NB20` download).


In [ ]:
sonar_f32 = sonar_features.astype(np.float32)

print(f"float64: {sonar_features.nbytes / 1024:.1f} KB")
print(f"float32: {sonar_f32.nbytes / 1024:.1f} KB")
print(f"Values still close after the precision drop? {np.allclose(sonar_features, sonar_f32, atol=1e-6)}")


One more common dtype trap, worth seeing once directly: **integer division truncates, and integer arrays can silently overflow their range** in ways float arrays don't. Neither of these will raise an error.


In [ ]:
int_array = np.array([1, 2, 3], dtype=np.int32)
float_array = np.array([1, 2, 3], dtype=np.float64)

print("Integer array / 2:", int_array / 2)      # NumPy promotes to float on true division -- safe
print("Integer array dtype after /:", (int_array / 2).dtype)

small_int8 = np.array([120], dtype=np.int8)      # int8 range: -128 to 127
print("\nint8 value:", small_int8[0])
print("int8 value + 20 (overflows silently):", (small_int8 + np.int8(20))[0])


> **Further reading**: [NumPy dtype documentation](https://numpy.org/doc/stable/reference/arrays.dtypes.html) | [Integer overflow (Wikipedia)](https://en.wikipedia.org/wiki/Integer_overflow)


---

## 3. Array creation patterns


`Real code rarely types out every array element by hand.` These creation functions cover most real situations: placeholders to fill in later (`zeros`/`ones`/`empty`), regular sequences (`arange`/`linspace`), coordinate grids for anything spatial (`meshgrid` — the exact tool `NB18` and `NB20` use to build a lat/lon grid from a real weather or bathymetry file), and identity matrices (`identity`, needed for `NB04`'s linear algebra).


In [ ]:
zeros_arr = np.zeros((3, 4))              # placeholder, e.g. "no readings yet" for 3 sensors x 4 timestamps
ones_arr = np.ones(5)                     # e.g. initial weights, all equal
arange_arr = np.arange(0, 24, 3)          # every 3rd hour in a 24-hour day
linspace_arr = np.linspace(0, 100, 5)     # 5 evenly-spaced depth levels from 0 to 100 m
identity_arr = np.identity(3)             # 3x3 identity matrix, needed in NB04

print("zeros:\n", zeros_arr)
print("\nones:", ones_arr)
print("\narange (every 3rd hour):", arange_arr)
print("\nlinspace (5 depth levels, 0-100 m):", linspace_arr)
print("\nidentity:\n", identity_arr)


`meshgrid` deserves its own example: it turns two 1-D coordinate arrays into two 2-D grids, one holding every point's x-coordinate and one holding every point's y-coordinate — exactly what `NB18` needed to turn ERA5's separate latitude/longitude arrays into a flat table of (lat, lon) pairs, and what `NB20` needed for the GMRT bathymetry grid.


In [ ]:
lon_1d = np.array([-1.0, -0.5, 0.0])
lat_1d = np.array([37.0, 37.5])

lon_grid, lat_grid = np.meshgrid(lon_1d, lat_1d)

print("1-D longitude:", lon_1d)
print("1-D latitude:", lat_1d)
print("\n2-D longitude grid (one row per latitude):\n", lon_grid)
print("\n2-D latitude grid (one column per longitude):\n", lat_grid)
print("\nEvery (lon_grid[i,j], lat_grid[i,j]) pair is one real grid point, e.g. point (0,0):",
      (lon_grid[0, 0], lat_grid[0, 0]))


> **Further reading**: [`numpy.meshgrid` documentation](https://numpy.org/doc/stable/reference/generated/numpy.meshgrid.html) | [`numpy.linspace` documentation](https://numpy.org/doc/stable/reference/generated/numpy.linspace.html)


---

## 4. Indexing and slicing, 1-D and multi-D


NumPy slicing extends Python's own `list[start:stop:step]` syntax `to multiple dimensions at once, with a comma separating each axis`. The cell below applies it directly to the real Sonar array (208 readings x 60 frequency bands).


In [ ]:
print("First reading, all 60 bands:", sonar_features[0].shape)
print("First 5 readings, first 3 bands:\n", sonar_features[:5, :3])
print("\nEvery other reading, last band:", sonar_features[::2, -1][:5], "...")
print("\nLast reading, reversed band order, first 5 values:", sonar_features[-1, ::-1][:5])


A frequent real mistake: assuming `array[i][j]` and `array[i, j]` always behave identically. For a plain 2-D NumPy array they give the same value, but `array[i][j]` does it in **two separate steps** (first slice out row `i` as its own array, then index into that) while `array[i, j]` does it in **one step** — a difference that matters for both speed and, as the next section shows, for whether you're looking at a view or a fresh copy.


In [ ]:
row_then_col = sonar_features[0][5]   # two steps: slice row 0, then index into it
direct = sonar_features[0, 5]         # one step: index both axes at once

print("Same value either way:", row_then_col == direct)
print("But array[0] alone is itself a real, separate array object:")
print("  type(sonar_features[0]):", type(sonar_features[0]))
print("  sonar_features[0].shape:", sonar_features[0].shape)


> **Further reading**: [NumPy indexing documentation](https://numpy.org/doc/stable/user/basics.indexing.html)


---

## 5. Views vs. copies (a real, common bug)


This is the single most common real NumPy bug for anyone new to it: **basic slicing (`array[a:b]`) returns a view — a window onto the *same* underlying memory, not a new array.** Modifying a view modifies the original. `np.array()` or `.copy()` on a slice makes a genuine, independent copy instead. `Neither the view nor the copy prints any warning about which one you have` — the only way to know is to check, or to know the rule.


In [ ]:
original = sonar_features[:5, :3].copy()   # protect the real data before this demo modifies anything
demo = original.copy()

view = demo[0]          # a VIEW: shares memory with `demo`
copy = demo[1].copy()   # a genuine COPY: independent memory

print("Before modification:")
print("demo[0]:", demo[0])
print("demo[1]:", demo[1])

view[0] = -999.0        # modifying the view...
copy[0] = -999.0        # ...and modifying the copy

print("\nAfter modifying `view` and `copy`:")
print("demo[0] (changed! `view` shared its memory):", demo[0])
print("demo[1] (unchanged -- `copy` was independent):", demo[1])


The diagram below makes the memory-sharing explicit: a view is a second *name* pointing at the same block of real memory, while a copy is an entirely separate block.


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

for ax, title, shares in zip(axes, ["View: demo[0]", "Copy: demo[1].copy()"], [True, False]):
    ax.add_patch(plt.Rectangle((0.05, 0.55), 0.4, 0.3, fill=True, facecolor="lightsteelblue", edgecolor="black"))
    ax.text(0.25, 0.7, "demo\n(memory block A)", ha="center", va="center", fontsize=9)
    if shares:
        ax.add_patch(plt.Rectangle((0.55, 0.55), 0.4, 0.3, fill=False, edgecolor="darkred", linewidth=2))
        ax.text(0.75, 0.7, "view\n(points into A)", ha="center", va="center", fontsize=9, color="darkred")
        ax.annotate("", xy=(0.55, 0.7), xytext=(0.45, 0.7), arrowprops=dict(arrowstyle="->", color="darkred"))
    else:
        ax.add_patch(plt.Rectangle((0.55, 0.15), 0.4, 0.3, fill=True, facecolor="mistyrose", edgecolor="darkred"))
        ax.text(0.75, 0.3, "copy\n(memory block B)", ha="center", va="center", fontsize=9, color="darkred")
        ax.annotate("", xy=(0.6, 0.4), xytext=(0.3, 0.55), arrowprops=dict(arrowstyle="->", color="gray", linestyle="--"))
        ax.text(0.45, 0.5, "data\ncopied", ha="center", fontsize=7, color="gray")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")
    ax.set_title(title, fontsize=10)

plt.suptitle("A view shares memory with the original; a copy does not")
plt.tight_layout()
plt.show()


**Practical rule**: whenever you slice an array and intend to modify the result *without* touching the original, call `.copy()` explicitly. `NB18`'s and `NB20`'s spatial grids, and every train/test split in this course, rely on this rule holding — a model accidentally trained on data corrupted by an unintended view-modification would be a very hard bug to trace back.

> **Further reading**: [NumPy copies and views documentation](https://numpy.org/doc/stable/user/basics.copies.html)


---

## 6. Boolean masks and fancy indexing on real data


A **boolean mask** is an array of `True`/`False` values, the same shape as the data, used to select elements by *condition* rather than by position — the tool behind every `df[df["column"] > x]`-style filter this course has used since `NB02`. The cell below applies it directly to the real Sonar labels.


In [ ]:
is_mine = sonar_labels == "M"
print(f"Real mine readings: {is_mine.sum()} / {len(sonar_labels)} ({is_mine.mean():.1%})")

mine_readings = sonar_features[is_mine]
rock_readings = sonar_features[~is_mine]

print(f"\nMean value of band 0, mines:  {mine_readings[:, 0].mean():.4f}")
print(f"Mean value of band 0, rocks:  {rock_readings[:, 0].mean():.4f}")


**Fancy indexing** takes this further: instead of a boolean mask, you pass an explicit *array of positions* to select (and can repeat or reorder them). The cell below uses it to pull out only 5 arbitrary frequency bands from every reading at once — the same mechanism `NB08`'s feature-importance ranking would use to pull out the *most informative* bands, once ranked.


In [ ]:
some_band_indices = np.array([10, 11, 35, 44, 48])  # illustrative subset of the 60 real frequency bands

subset_only = sonar_features[:, some_band_indices]
print(f"Full data shape: {sonar_features.shape}")
print(f"5-bands-only shape: {subset_only.shape}")
print("\nFirst reading, those 5 bands only:", subset_only[0])


One important difference from slicing: **fancy indexing always returns a copy, never a view** — since the selected elements aren't necessarily contiguous in memory, NumPy has no choice but to build a new array. `This is worth knowing precisely because it's the opposite of Section 5's rule for basic slicing`.


In [ ]:
fancy_result = sonar_features[[0, 1, 2]]
print("Fancy indexing shares memory with the original?", np.shares_memory(fancy_result, sonar_features))

slice_result = sonar_features[0:3]
print("Basic slicing shares memory with the original?", np.shares_memory(slice_result, sonar_features))


> **Further reading**: [NumPy boolean and fancy indexing documentation](https://numpy.org/doc/stable/user/basics.indexing.html#advanced-indexing) | [`numpy.shares_memory` documentation](https://numpy.org/doc/stable/reference/generated/numpy.shares_memory.html)


---

## Class summary

- `dtype` controls both memory usage and correctness (silent truncation, silent overflow) — always worth knowing, not just assuming NumPy's default is right for the task.
- Creation functions (`zeros`/`ones`/`arange`/`linspace`/`meshgrid`/`identity`) cover most real array-building needs without writing out elements by hand.
- Multi-dimensional indexing/slicing extends Python's own slice syntax with a comma per axis.
- **Basic slicing returns a view (shares memory); fancy indexing and `.copy()` return a genuine copy.** This one rule prevents a real, common, silent class of bug.
- Boolean masks and fancy indexing select real data by condition or by an explicit list of positions — the same tool underlying every filter this course has used since `NB02`.

## For the next class (NB04)

We go from single arrays to real matrix and vector operations — broadcasting, `einsum`, and solving real systems of linear equations with `numpy.linalg`, applied to a genuine naval statics problem (mooring-line tensions).

## Homework / Practice Ideas

1. Load `ship_fuel_efficiency.csv` (`NB07`) as a NumPy array (not a DataFrame) and use boolean masking to select only rows where `ship_type == "Tanker Ship"`.
2. Demonstrate the view/copy distinction yourself: slice out a 2-column subset of any array here, modify it, and check with `np.shares_memory` whether the original changed.
3. Use `np.meshgrid` to build a 5x5 grid of (x, y) coordinates spanning -2 to 2 in both directions, then compute `z = x**2 + y**2` on the whole grid at once (no loop).
4. Compare the real memory footprint (`.nbytes`) of the Sonar dataset stored as `float64`, `float32`, and `float16` — at what point (if any) does `np.allclose` start reporting real numerical differences?
5. Using fancy indexing, build a "shuffled" copy of the Sonar dataset's row order (hint: `np.random.permutation`) and confirm with `np.shares_memory` that it is a genuine copy, not a view.

> ***As always: a real dataset already used elsewhere in this course is exactly what makes an array-mechanics exercise concrete instead of abstract.***
